In [1]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
import re
import nltk
from nltk.corpus import stopwords
import pymorphy3
import pandas as pd

In [3]:
# Для русских текстов
morph = pymorphy3.MorphAnalyzer(lang='ru')
stop_words = set(stopwords.words('russian'))

def preprocess_text(text):
    # Удаление цензурных символов (****, ### и т.д.)
    text = re.sub(r'[*#]+', '', text)
    
    # Удаление спецсимволов, но сохранение основных знаков препинания
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s.,!?-]', ' ', text)
    
    # Приведение к нижнему регистру
    text = text.lower()
    
    # Токенизация
    tokens = text.split()
    
    # Удаление стоп-слов и лемматизация
    cleaned_tokens = []
    for token in tokens:
        if token not in stop_words and len(token) > 2:
            lemma = morph.parse(token)[0].normal_form
            cleaned_tokens.append(lemma)
    
    return ' '.join(cleaned_tokens)

In [4]:
table_data = pd.read_excel("../../data_raw/DataSet_V49 (2).xlsx")

In [5]:
text_data = pd.read_csv("../../anonymized_parsed_data_histories.csv")

In [6]:
len(text_data)

21670

In [7]:
table_data.dropna(subset=['Смерть'], inplace=True)

In [8]:
len(table_data)

16613

In [9]:
df2_subset = table_data[['Файл(ИБ)', 'Смерть']]
df2_subset.rename(columns={'Файл(ИБ)': 'file_name', 'Смерть': 'label'}, inplace=True)

df2_subset["file_name"] = df2_subset["file_name"].astype(str)
# text_data['file_name'] = text_data['file_name'].apply(lambda x: str(int(x)) if pd.notna(x) else None)

# Делаем merge по идентификатору
df1 = text_data.merge(df2_subset, on='file_name', how='inner')

/tmp/ipykernel_2681605/2526386153.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_subset.rename(columns={'Файл(ИБ)': 'file_name', 'Смерть': 'label'}, inplace=True)
/tmp/ipykernel_2681605/2526386153.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_subset["file_name"] = df2_subset["file_name"].astype(str)


In [10]:
df1

,file_name,patient_id,anonymized_text_data,label
0,ДружковаГВ25177_history_0.html,25177.0,[ФИО] «Приморская краевая клиническая больница...,Нет
1,ДорошенкоВИ_history_0.html,7461.0,[ФИО] «Приморская краевая клиническая больница...,Нет
2,ТкаченкоВВ278_history_0.html,278.0,[ФИО] «Приморская краевая клиническая больница...,Нет
3,ЗубоваИН9275_history_0.html,9275.0,[ФИО] «Приморская краевая клиническая больница...,Нет
4,НесинаНД_history_0.html,4039.0,[ФИО] «Приморская краевая клиническая больница...,Нет
...,...,...,...,...
10453,ДовшковаЛФ_history_0.html,8567.0,[ФИО] «Приморская краевая клиническая больница...,Нет
10454,АхапкинАВ_history_0.html,13287.0,[ФИО] «Приморская краевая клиническая больница...,Нет
10455,КонстантиновКВ4084_history_0.html,4084.0,[ФИО] «Приморская краевая клиническая больница...,Нет
10456,ЛысенкоТВ9514_history_0.html,9514.0,[ФИО] «Приморская краевая клиническая больница...,Нет


In [11]:
df1['anonymized_text_data'][0]

'[ФИО] «Приморская краевая клиническая больница №1» (690990, Владивосток, ул. Алеутская 57) Согласие пациента на обработку персональных данных. Я, (\nФ.И.О. полностью) [ФИО] Место рождения [ФИО] край, г.Уссурийск, ул. Октябрьская д.5 кв.56 Паспорт [ПАСПОРТНЫЕ ДАННЫЕ] , выдан 25.07.2006 Орган, выдавший паспорт Даю своё согласие на обработку своих персональных данных [ФИО] Здравоохранения «Приморской краевой клинической больнице №1 » в целях сохранения моего здоровья на срок в соответствии с действующим законодательством. Персональные данные, необходимые для использования в ГУЗ ПККБ №1 – фамилия, имя, отчество, год, месяц,\nдата рождения, адрес, семейное, социальное положение, образование, профессия, медицинский\nдиагноз, сведения о перенесенных заболеваниях, а так же другие сведения необходимые для постановки\nдиагноза и проведения лечения. Обработка персональных данных включает в себя сбор, систематизацию, накопление, хранение, уточнение (обновление, изменение), использование, передачу

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec, FastText

In [13]:
df1['anonymized_text_data'] = df1['anonymized_text_data'].apply(lambda x: preprocess_text(x))

In [58]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 3),
    analyzer='word',
    sublinear_tf=True
)

X_texts = vectorizer.fit_transform(df1["anonymized_text_data"])

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

In [27]:
df1["processed"] = vectorizer.fit_transform(df1["anonymized_text_data"])

TypeError: sparse array length is ambiguous; use getnnz() or shape[0]

In [59]:
print()

In [33]:
df1["label"] = df1["label"].map({'Да': 1, 'Нет': 0})

In [60]:
X_train_full, X_val, y_train_full, y_val = train_test_split(
    X_texts, df1["label"], test_size=0.2, random_state=42, stratify=df1["label"]
)

In [61]:
X_train, X_test, y_train, y_test = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X_texts, df1["label"], test_size=0.2, random_state=42, stratify=df1["label"]
)


In [25]:
df_train

,file_name,patient_id,anonymized_text_data,label
6406,НикитенкоВП_history_0.html,19932.0,фио приморский краевой клинический больница 69...,Нет
909,МореваВН12270_history_0.html,12270.0,фио приморский краевой клинический больница 69...,Нет
8958,МисюринаАП_history_0.html,20526.0,фио приморский краевой клинический больница 69...,Нет
7326,ХильТА11840_history_0.html,11840.0,фио приморский краевой клинический больница 69...,Нет
5002,МолочинаОВ12871_history_0.html,12871.0,фио приморский краевой клинический больница 69...,Нет
...,...,...,...,...
8798,ГригорянМР_history_0.html,7578.0,фио приморский краевой клинический больница 69...,Нет
5574,ЧечельЕП_history_0.html,21324.0,фио приморский краевой клинический больница 69...,Нет
2702,ЮнРН4055_history_0.html,4055.0,фио приморский краевой клинический больница 69...,Нет
3671,ШишкинВК_history_0.html,12819.0,фио приморский краевой клинический больница 69...,Нет


In [37]:
y_val

8366    0
6802    0
4816    0
1012    0
9585    0
       ..
2221    0
4941    0
4800    0
481     0
9730    0
Name: label, Length: 2092, dtype: int64

In [41]:
y_val

8366    0
6802    0
4816    0
1012    0
9585    0
       ..
2221    0
4941    0
4800    0
481     0
9730    0
Name: label, Length: 2092, dtype: int64

In [62]:
text_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

text_model.fit(X_train, y_train)

y_pred_proba = text_model.predict_proba(X_val)[:, 1]
y_pred = text_model.predict(X_val)
auc = roc_auc_score(y_val, y_pred_proba)
print(f"AUC на val: {auc:.4f}")

AUC на val: 0.9988


In [63]:
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      2003
           1       0.83      0.98      0.90        89

    accuracy                           0.99      2092
   macro avg       0.91      0.98      0.95      2092
weighted avg       0.99      0.99      0.99      2092



In [64]:
y_pred_proba = text_model.predict_proba(X_test)[:, 1]
y_pred = text_model.predict(X_test)
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC на val: {auc:.4f}")

AUC на val: 0.9973


In [65]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1603
           1       0.87      0.94      0.91        71

    accuracy                           0.99      1674
   macro avg       0.93      0.97      0.95      1674
weighted avg       0.99      0.99      0.99      1674



In [49]:
text_model.coef_

array([[ 0.05871388, -0.00336014, -0.11952745, ...,  0.32387704,
         0.12455201,  0.12455201]], shape=(1, 10000))

In [66]:
feature_names = vectorizer.get_feature_names_out()

In [56]:
import numpy as np

In [68]:
coefs = text_model.coef_[0]  # для бинарной классификации

# индексы сортировки по абсолютной важности
top_indices = np.argsort(np.abs(coefs))[::-1][:50]  # топ-20

# печатаем топ
for idx in top_indices:
    print(f"{feature_names[idx]}: {coefs[idx]:.4f}")

мероприятие: 3.7956
адреналин: 3.0517
реанимационный: 2.5957
отметить: 2.1852
катетеризация: 2.0405
прекратить: 1.9414
подключичный: 1.8158
центральный: 1.8068
катетеризация подключичный: 1.7783
катетеризация подключичный другой: 1.7759
подключичный другой: 1.7759
другой центральный: 1.7759
подключичный другой центральный: 1.7759
другой центральный вено: 1.7749
центральный вено: 1.7443
шок: 1.6105
минут: 1.5468
кардиогенный шок: 1.5216
стенка левый желудочек: 1.5077
норадреналин: 1.4662
стенка левый: 1.4611
др: 1.3988
sol morphini hydrochloridi: 1.3772
hydrochloridi: 1.3771
morphini hydrochloridi: 1.3771
sol morphini: 1.3761
morphini: 1.3755
биологический: 1.2667
morphini hydrochloridi ml: 1.2068
hydrochloridi ml: 1.2068
фио sol morphini: 1.1959
ketamini: 1.1817
sol ketamini: 1.1817
номер история: 1.1756
цель седации: 1.1738
седации: 1.1497
центральный вено рсц: 1.1321
вено рсц: 1.1321
свидетельство рождение: 1.1315
ребёнок: 1.1128
зонд: 1.1086
hydrochloridi ml один: 1.0482
медицинский

In [28]:
count = 0
for i in sorted(y_pred):
    if i == 'Да':
        count += 1

print(count)

126


In [ ]:
print("Hello world")

In [ ]:
y_pred_proba = text_model.predict_proba(df_test["processed"])[:, 1]
y_pred = text_model.predict(df_test["processed"])
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC на val: {auc:.4f}")

In [ ]:
print(classification_report(y_test, y_pred))